# Persistence, Durability & Tasks

LangGraph has **two persistence systems**:

1. **Checkpointers** - short-term, thread-scoped snapshots of graph execution
2. **Stores** - long-term, cross-thread memory shared across workflows

Persistence matters because it enables:

- **Conversation continuity** across turns
- **Fault tolerance** after crashes or interrupts
- **Human-in-the-loop (HITL)** pause / review / resume patterns
- **Time-travel** debugging, replay, and branching from earlier checkpoints

In this notebook we will focus on **checkpointers**, **durability modes**, and the **`@task` decorator**.


In [ ]:
%run ../../langchain_common.py

## Checkpointer Deep Dive

A **checkpoint** is a snapshot of graph state saved after each super-step.  
For a simple flow like **START -> A -> B -> END**, LangGraph creates **4 checkpoints**:

1. Initial input at **START**
2. State after **A**
3. State after **B**
4. Final terminal snapshot at **END**

`MemorySaver` is the simplest checkpointer for demos because it keeps everything in memory for the current Python process.


In [ ]:
from pprint import pprint
from typing import Annotated, TypedDict
import operator

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph


class DemoState(TypedDict, total=False):
    topic: str
    steps: Annotated[list[str], operator.add]
    draft: str
    final: str


def node_a(state: DemoState):
    return {
        "steps": ["A"],
        "draft": f"Draft about: {state['topic']}",
    }


def node_b(state: DemoState):
    return {
        "steps": ["B"],
        "final": f"{state['draft']} -> polished by B",
    }


builder = StateGraph(DemoState)
builder.add_node("a", node_a)
builder.add_node("b", node_b)
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)
config = {"configurable": {"thread_id": "persistence-demo"}}


In [ ]:
input_state = {
    "topic": "why persistence matters",
    "steps": [],
}

for step_index, state in enumerate(graph.stream(input_state, config, stream_mode="values")):
    print(f"Super-step {step_index}")
    pprint(state)
    print("-" * 70)

history = list(graph.get_state_history(config))
print(f"Saved checkpoints: {len(history)}")


## State History & Time Travel

Every checkpoint can be inspected later with `graph.get_state_history(config)`.  
This is useful for:

- auditing what happened
- debugging a bad run
- replaying from an earlier checkpoint
- **forking** by editing saved state and continuing from there

`get_state_history()` usually returns the **newest checkpoint first**, so reversing it makes the timeline easier to read.


In [ ]:
history = list(graph.get_state_history(config))
history_chrono = list(reversed(history))

for index, snapshot in enumerate(history_chrono, start=1):
    checkpoint_id = snapshot.config["configurable"].get("checkpoint_id")
    print(f"Checkpoint {index}: {checkpoint_id}")
    print("values =", snapshot.values)
    print("next   =", snapshot.next)
    print()


In [ ]:
after_a = next(snapshot for snapshot in history if snapshot.next == ("b",))
replay_config = {
    "configurable": {
        "thread_id": config["configurable"]["thread_id"],
        "checkpoint_id": after_a.config["configurable"]["checkpoint_id"],
    }
}

replayed = graph.invoke(None, replay_config)
print("Replayed from the checkpoint immediately after A:")
pprint(replayed)


In [ ]:
fork_config = graph.update_state(
    replay_config,
    {"draft": "Forked draft about durability and crash recovery"},
    as_node="a",
)

forked = graph.invoke(None, fork_config)
print("Forked result after changing the saved state from A:")
pprint(forked)


## Built-in Checkpointers

| Checkpointer | Best for | Notes |
|---|---|---|
| `MemorySaver` | Local demos and teaching | In-memory only; disappears when the process stops |
| `SqliteSaver` | Lightweight persistence on one machine | File-based; convenient for prototypes and single-user apps |
| `PostgresSaver` | Production workloads | Durable, concurrent, and suitable for multi-worker deployments |

In this notebook we used **`MemorySaver`** because it is the easiest way to see checkpoint behavior.  
The SQLite and Postgres options follow the same core idea, but store snapshots in a durable backend.


## Durability Modes

Durability controls **when** a checkpoint is persisted:

| Mode | Meaning | Trade-off |
|---|---|---|
| `exit` | Save only when the run finishes, errors, or interrupts | Fastest, but least durable |
| `async` | Save in the background while the next step runs | Good default balance |
| `sync` | Save before moving to the next step | Strongest durability, highest latency |

Some older examples show:

```python
graph.invoke(input, config, checkpoint_during="async")
```

In the current LangGraph API used here, the equivalent keyword is:

```python
graph.invoke(input, config, durability="async")
```


In [ ]:
durability_config = {"configurable": {"thread_id": "durability-demo"}}

async_result = graph.invoke(
    {"topic": "durability modes", "steps": []},
    durability_config,
    durability="async",
)

pprint(async_result)


## The `@task` Decorator

In LangGraph's Functional API, `@task` wraps **side-effectful** or **non-deterministic** work so LangGraph can reuse the saved result during replay.

```python
from langgraph.func import task
import requests

@task
def call_api(url: str) -> str:
    return requests.get(url).text[:100]
```

Why this matters:

- Without `@task`, replaying from a checkpoint can repeat the API call
- With `@task`, LangGraph can reuse the saved task result instead of doing the side effect again

Typical uses: API calls, timestamps, randomness, database writes, file writes, or payment / email actions.


In [ ]:
from datetime import UTC, datetime

from langgraph.func import entrypoint, task


task_runs = []


@task
def fetch_nonce(label: str) -> str:
    stamp = f"{label}@{datetime.now(UTC).isoformat(timespec='seconds')}"
    task_runs.append(stamp)
    return stamp


@entrypoint(checkpointer=MemorySaver())
def task_demo(_: dict):
    nonce = fetch_nonce("api-call").result()
    explanation = llm_noreason.invoke(
        "Explain in one short sentence why a replayed workflow should reuse "
        f"the saved side-effect result '{nonce}' instead of recomputing it."
    ).content
    return {"nonce": nonce, "explanation": explanation}


task_config = {"configurable": {"thread_id": "task-demo"}}
first = task_demo.invoke({}, task_config)
second = task_demo.invoke(None, task_config)

print("Task executions recorded:", len(task_runs))
print("Same saved nonce returned on replay:", first["nonce"] == second["nonce"])
print(second["explanation"])


## Key Takeaways

| Topic | Options / Pattern | When to use |
|---|---|---|
| Checkpointer backend | `MemorySaver` | Demos, notebooks, local development |
| Checkpointer backend | `SqliteSaver` | Lightweight persistence on a single machine |
| Checkpointer backend | `PostgresSaver` | Production systems with concurrency and durability |
| Durability | `exit` | Fastest runs when losing intermediate steps is acceptable |
| Durability | `async` | Good default for many apps |
| Durability | `sync` | Maximum safety before each next step |
| `@task` | Wrap side effects / non-determinism | Prevent duplicate API calls, timestamps, writes, or other repeated actions during replay |

One final distinction:

- **Checkpointers** remember what happened inside one thread of execution
- **Stores** keep longer-lived memory that can be shared across threads and sessions
